In [ ]:
!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.23.5 --force-reinstall # numpyのバージョンを修正し、強制的に再インストール
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    IN_COLAB = False
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import sys
from pathlib import Path
from IPython.display import HTML
from base64 import b64encode
import json

sys.path.insert(0, '.')

from src.pipelines.player_pose_exporter import PlayerPoseExporter
from src.pipelines import (
    PipelineConfig,
    TableDetectionConfig,
    PoseTrackingConfig,
    PlayerClassificationConfig,
    TrackingExportConfig,
    VideoProcessingConfig
)

print("✓ モジュールのインポートが完了しました")

In [ ]:
INPUT_VIDEO = 'data/raw/sample_video_11_01.MOV'
OUTPUT_VIDEO = 'data/detect/sample_video_11_01/player_classification_result.mp4'
CSV_OUTPUT = 'data/detect/sample_video_11_01/player_pose_data.csv' 

In [ ]:
with open("configs/pipeline_config.json", "r") as f:
    config_dict = json.load(f)

config = PipelineConfig(
    table_detection=TableDetectionConfig(**config_dict['table_detection']),
    pose_tracking=PoseTrackingConfig(**config_dict['pose_tracking']),
    player_classification=PlayerClassificationConfig(**config_dict['player_classification']),
    tracking_export=TrackingExportConfig(**config_dict['tracking_export']),
    video_processing=VideoProcessingConfig(**config_dict['video_processing'])
)

In [ ]:
exporter = PlayerPoseExporter(config)

results = exporter.process_video(
    input_video=INPUT_VIDEO,
    output_video=OUTPUT_VIDEO,
    csv_output=CSV_OUTPUT
)